In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame named 'forrest'
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')

# Display the first few rows of the DataFrame to verify the data has been loaded
print(train.head())

In [ ]:
train.info()

In [ ]:
#check if there is any class imbalancing
train['Cover_Type'].value_counts(normalize= True)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

scaler = StandardScaler()

# Extract features and target variable
X = train.drop(['Id', 'Cover_Type'], axis=1)
y = train['Cover_Type']

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
import time

# Assuming X and y are your features and labels

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Apply PCA
pca = PCA(n_components=0.95)  # You can adjust the explained variance threshold
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

# Define the parameter grid for KNN
knn_param_grid = {'n_neighbors': [3, 5, 7, 9],
                  'weights': ['uniform', 'distance'],
                  'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}

# Train KNN model with GridSearchCV for parameter tuning
knn_model = KNeighborsClassifier()
grid_knn = GridSearchCV(estimator=knn_model, param_grid=knn_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
start_time = time.time()
grid_knn.fit(X_train_pca, y_train)
training_time = time.time() - start_time
print("Training Time: {:.2f} seconds".format(training_time))

# Print the best parameters
print("Best Parameters for KNN: ", grid_knn.best_params_)

# Make predictions on validation set
start_time = time.time()
y_pred_val = grid_knn.predict(X_val_pca)
prediction_time_val = time.time() - start_time
print("Validation Prediction Time: {:.2f} seconds".format(prediction_time_val))

# Evaluate accuracy on validation set
accuracy_val = accuracy_score(y_val, y_pred_val)
print("Validation Accuracy: ", accuracy_val)


In [ ]:
# Plot training & validation accuracy values for the last fold
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(accuracy_val, label='Validation Accuracy')
plt.title('Model accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

**Submission**

In [ ]:
import pandas as pd

# Load the test set into a DataFrame
test_set = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

# Display the first few rows of the test set
print(test_set.head())

In [ ]:
X_test = test_set.drop(['Id'], axis=1)

# Assuming X_test is your test set
X_test_scaled = scaler.transform(X_test.values)

# Apply PCA on the test set
X_test_scaled_pca = pca.transform(X_test_scaled)

# Make predictions on the test set
start_time = time.time()
predictions = grid_knn.predict(X_test_scaled_pca)
prediction_time_test = time.time() - start_time
print("Test Prediction Time: {:.2f} seconds".format(prediction_time_test))

# Create a DataFrame with the observation Id and predicted cover type
submission_df = pd.DataFrame({'Id': test_set['Id'], 'Cover_Type': predictions})

# Save the DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)